# Part 2: Vectorisation, batching, and broadcasting

A GPU can execute many arithmetic operations concurrently, but only when we express the work as suitably large tensor operations. This notebook develops one matrix multiplication from scalar loops into a batched PyTorch operation, checks that the implementations agree, and then benchmarks them.

In [ ]:
import torch
import torch.utils.benchmark as benchmark

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; selected device: {device}")

## Matrix multiplication from scalar operations

For $A \in \mathbb{R}^{m\times k}$ and $B \in \mathbb{R}^{k\times n}$, the element in row $i$ and column $j$ of $C=AB$ is

$$C_{ij} = \sum_{r=1}^{k} A_{ir}B_{rj}.$$

### Task: implement the definition

Complete `matmul_scalar_loops`. Your implementation should use three Python loops and individual tensor elements. Create the result with the same dtype and device as `a`.

In [ ]:
def matmul_scalar_loops(a, b):
    assert a.ndim == 2 and b.ndim == 2
    assert a.shape[1] == b.shape[0]

    # TODO: implement the indexed definition above.
    raise NotImplementedError


In [ ]:
a_small = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
b_small = torch.tensor([[2.0, 1.0], [0.0, 3.0], [1.0, -1.0]])

expected = a_small @ b_small
actual = matmul_scalar_loops(a_small, b_small)

print(actual)
torch.testing.assert_close(actual, expected)
print("Task 1 checks passed")

The loop implementation is valuable because it exposes the mathematics. It is not how we should normally ask PyTorch to perform the computation. `a @ b`, `torch.matmul(a, b)`, and for two matrices `torch.mm(a, b)` dispatch the work to compiled numerical kernels.

## Adding a batch axis

Now let

- `a_batch` have shape `(batch, m, k)`, and
- `b_batch` have shape `(batch, k, n)`.

We want one matrix product for each corresponding pair in the batch, producing `(batch, m, n)`.

### Task: loop over the batch

Complete `batch_matmul_loop`. This time use one Python loop over the batch and the built-in `@` operator inside it. Do not call your scalar-loop implementation.

In [ ]:
def batch_matmul_loop(a, b):
    assert a.ndim == 3 and b.ndim == 3
    assert a.shape[0] == b.shape[0]
    assert a.shape[2] == b.shape[1]

    # TODO
    raise NotImplementedError


In [ ]:
a_batch = torch.randn(8, 5, 7)
b_batch = torch.randn(8, 7, 3)

loop_result = batch_matmul_loop(a_batch, b_batch)
bmm_result = torch.bmm(a_batch, b_batch)

assert loop_result.shape == (8, 5, 3)
torch.testing.assert_close(loop_result, bmm_result)
print("Task 2 checks passed")

`torch.bmm` performs a batch of matrix–matrix products. It does not broadcast its batch dimension. `torch.matmul` is more general and can broadcast compatible leading dimensions. That generality is useful, but it also makes careful shape reasoning important.

## Broadcasting: predict before running

PyTorch compares dimensions from right to left. Two dimensions are compatible when they are equal, one is `1`, or one does not exist. Missing leading dimensions are treated as dimensions of size one.

For each pair below, decide whether it broadcasts. If it does, write down the output shape.

| Left shape | Right shape | Broadcasts? | Output shape |
|---|---|---|---|
| `(32, 128)` | `(128,)` | ? | ? |
| `(32, 128)` | `(32,)` | ? | ? |
| `(8, 1, 64)` | `(1, 16, 1)` | ? | ? |
| `(4, 3, 1)` | `(3, 5)` | ? | ? |

In [ ]:
shape_pairs = [
    ((32, 128), (128,)),
    ((32, 128), (32,)),
    ((8, 1, 64), (1, 16, 1)),
    ((4, 3, 1), (3, 5)),
]

for left, right in shape_pairs:
    try:
        print(left, right, "->", torch.broadcast_shapes(left, right))
    except RuntimeError as error:
        print(left, right, "-> incompatible")

### Task: scale each example

`samples` contains 12 examples with 20 features. `scales` contains one scale factor per example. The expression `samples * scales` is not valid because PyTorch first tries to align the length-12 axis with the length-20 trailing axis.

Use `unsqueeze`, indexing with `None`, or `reshape` to make the intended alignment explicit.

In [ ]:
samples = torch.randn(12, 20)
scales = torch.linspace(0.5, 1.5, 12)

# TODO
scaled_samples = None

assert scaled_samples.shape == samples.shape
torch.testing.assert_close(scaled_samples[0], samples[0] * scales[0])
torch.testing.assert_close(scaled_samples[-1], samples[-1] * scales[-1])
print("Task 3 checks passed")

### Batched `matmul` with shared weights

A common deep-learning operation applies the same weight matrix to every item in a batch. If `sequences` has shape `(batch, tokens, input_features)` and `weights` has shape `(input_features, output_features)`, `torch.matmul` treats the leading axes as batch-like axes.

In [ ]:
sequences = torch.randn(16, 25, 64)       # (batch, tokens, input features)
weights = torch.randn(64, 128)            # (input features, output features)
bias = torch.randn(128)                   # (output features,)

projected = sequences @ weights + bias
print(projected.shape)
assert projected.shape == (16, 25, 128)

Explain which operation treats the first two axes as batch-like axes, and which operation broadcasts over them.

## Benchmarking your implementations

A useful benchmark performs warm-up work, repeats the measurement, and synchronises asynchronous accelerator operations. `torch.utils.benchmark.Timer` handles these details for PyTorch operations. We will report medians rather than trusting one run.

First compare the scalar-loop definition with the built-in operation on the CPU. Keep this example deliberately small: Python executes every loop iteration.

In [ ]:
a_timing = torch.randn(24, 24)
b_timing = torch.randn(24, 24)

loop_time = benchmark.Timer(
    stmt="matmul_scalar_loops(a_timing, b_timing)",
    globals=globals(),
).blocked_autorange(min_run_time=0.2)

builtin_time = benchmark.Timer(
    stmt="a_timing @ b_timing",
    globals=globals(),
).blocked_autorange(min_run_time=0.2)

print(loop_time)
print(builtin_time)
print(f"Built-in speed-up: {loop_time.median / builtin_time.median:.1f}×")

Now compare a Python batch loop with `torch.bmm`. These operations can run on either the CPU or the selected accelerator.

In [ ]:
a_large = torch.randn(64, 64, 64, device=device)
b_large = torch.randn(64, 64, 64, device=device)

loop_batch_time = benchmark.Timer(
    stmt="batch_matmul_loop(a_large, b_large)",
    globals=globals(),
).blocked_autorange(min_run_time=0.5)

bmm_time = benchmark.Timer(
    stmt="torch.bmm(a_large, b_large)",
    globals=globals(),
).blocked_autorange(min_run_time=0.5)

print(loop_batch_time)
print(bmm_time)
print(f"Batched-operation speed-up: {loop_batch_time.median / bmm_time.median:.1f}×")

### Task: investigate scale

Repeat the `torch.bmm` benchmark for at least three different batch sizes or matrix sizes. Record the median times and compute throughput in products per second.

If a GPU is available, repeat one comparison on both CPU and GPU. Do not assume the GPU will win for a small problem. Explain the result in terms of fixed overhead and the amount of parallel work available.

In [ ]:
# TODO: record your experiment in a small list of dictionaries, then print it.
results = []

## Wrap-up challenge: batched linear transformation

Implement a function computing

$$Y_{b,t,:}=X_{b,t,:}W+b$$

for a batch of sequences. Use one matrix multiplication and one broadcasted addition with no explicit Python loops. Include shape checks.

In [ ]:
def batched_linear(x, weight, bias):
    # x:      (batch, items, input_features)
    # weight: (input_features, output_features)
    # bias:   (output_features,)
    # TODO
    raise NotImplementedError


x_test = torch.randn(7, 11, 5)
w_test = torch.randn(5, 3)
b_test = torch.randn(3)
y_test = batched_linear(x_test, w_test, b_test)

assert y_test.shape == (7, 11, 3)
torch.testing.assert_close(y_test[2, 4], x_test[2, 4] @ w_test + b_test)
print("Consolidation checks passed")

## Reflection

Write brief answers to the following questions.

1. Why is the scalar-loop implementation useful pedagogically but unsuitable for training a model?
2. What work did we vectorise when moving from the scalar loops to `@`?
3. What additional work did we batch when moving from the batch loop to `torch.bmm`?
4. Why can two mathematically equivalent floating-point implementations differ by a small amount?
5. Why might a GPU lose to a CPU on a small tensor operation?
6. Give one example where broadcasting silently produces a valid tensor with the wrong shape.